# Module 2: RAG Systems (Retrieval-Augmented Generation)

**Goal:** Build a working RAG pipeline from scratch — from documents to answerable questions.

**What you'll learn:**
1. Why RAG exists and when to use it
2. Document chunking strategies (fixed-size vs. semantic)
3. Creating and querying a vector store (ChromaDB)
4. Building a complete Q&A chain with source citations
5. Evaluating RAG quality

**Prerequisites:** Module 01 completed. `.env` file with `OPENAI_API_KEY`.

---

## Why RAG?

LLMs have a **knowledge cutoff** and no access to your private data. RAG solves this by:

```
User question
     │
     ▼
[Retriever] ──searches──▶ Vector DB (your documents)
     │                         │
     │◀──── top-k chunks ──────┘
     │
     ▼
[Generator] ──reads──▶ LLM (context + question → answer)
     │
     ▼
Answer + sources
```

**Use RAG when:** domain-specific knowledge, current events, private documents, reducing hallucinations.

## 0. Setup

In [ ]:
%pip install -q openai langchain langchain-community langchain-openai chromadb sentence-transformers tiktoken python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in ../.env"
print("✅ Environment ready")

---
## 1. Documents: Our Knowledge Base

In production these come from PDFs, databases, web pages, etc. Here we use in-memory documents.

In [ ]:
documents = [
    {
        "id": "rag_intro",
        "content": """Retrieval-Augmented Generation (RAG) combines information retrieval with 
        large language models. It retrieves relevant documents from a knowledge base and 
        provides them as context to the LLM, enabling more accurate and grounded responses. 
        RAG was introduced by Lewis et al. in 2020 and has become the dominant architecture 
        for knowledge-intensive NLP tasks.""",
        "metadata": {"source": "rag_overview", "topic": "fundamentals"}
    },
    {
        "id": "vector_db",
        "content": """Vector databases store embeddings — dense numerical representations of text. 
        When you search, your query is also embedded and the database returns the most 
        semantically similar stored vectors using approximate nearest-neighbor (ANN) search. 
        Popular options: ChromaDB (local), Pinecone (managed), Weaviate (open-source), 
        pgvector (PostgreSQL extension).""",
        "metadata": {"source": "vector_db_guide", "topic": "infrastructure"}
    },
    {
        "id": "chunking",
        "content": """Chunking splits documents into smaller pieces before embedding. 
        Fixed-size chunking splits every N tokens regardless of content boundaries. 
        Semantic chunking splits at natural boundaries (sentences, paragraphs, sections). 
        Chunk size matters: too small loses context, too large dilutes relevance. 
        A common starting point is 512 tokens with 10% overlap between chunks.""",
        "metadata": {"source": "chunking_guide", "topic": "preprocessing"}
    },
    {
        "id": "embeddings",
        "content": """Embeddings are vectors that capture semantic meaning. Two sentences about 
        the same topic have similar (high cosine similarity) embeddings even if they use 
        different words. Popular embedding models: OpenAI text-embedding-3-small (fast, cheap), 
        text-embedding-3-large (best quality), sentence-transformers/all-MiniLM-L6-v2 (free, local).""",
        "metadata": {"source": "embeddings_guide", "topic": "fundamentals"}
    },
    {
        "id": "advanced_rag",
        "content": """Advanced RAG techniques include: HyDE (Hypothetical Document Embeddings) 
        which generates a hypothetical answer and uses that for retrieval; Re-ranking which 
        retrieves more candidates then reranks with a cross-encoder; Hybrid search which 
        combines BM25 keyword search with vector similarity; Query expansion which rewrites 
        the question multiple ways to improve recall.""",
        "metadata": {"source": "advanced_rag", "topic": "advanced"}
    },
]

print(f"Loaded {len(documents)} documents")
print(f"Example document: '{documents[0]['content'][:100]}...'")

---
## 2. Chunking

Compare fixed-size vs. recursive character splitting. **Overlap** ensures context isn't lost at chunk boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain.schema import Document as LangchainDoc

# Convert to LangChain documents
lc_docs = [
    LangchainDoc(page_content=doc["content"], metadata=doc["metadata"])
    for doc in documents
]

# Strategy 1: Fixed-size chunks
fixed_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator=" ")
fixed_chunks = fixed_splitter.split_documents(lc_docs)

# Strategy 2: Recursive (respects sentence/paragraph boundaries)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.split_documents(lc_docs)

print(f"Original docs: {len(lc_docs)}")
print(f"Fixed-size chunks: {len(fixed_chunks)}")
print(f"Recursive chunks: {len(recursive_chunks)}")
print(f"\nExample chunk (recursive):\n{recursive_chunks[0].page_content}")

In [ ]:
# 🧪 EXERCISE: Experiment with chunk_size values: 100, 500, 1000
# Print chunk counts and check if sentences are split mid-way.
# Which strategy preserves meaning better?

---
## 3. Vector Store (ChromaDB)

Embed all chunks and store them. ChromaDB stores to disk by default — no server needed for development.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

# Clean up any previous run
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build vector store from recursive chunks
vectorstore = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="rag_tutorial",
)

print(f"✅ Vector store created with {vectorstore._collection.count()} vectors")

---
## 4. Retrieval: Finding Relevant Chunks

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # return top 3 chunks

query = "What are vector databases and which ones are popular?"
results = retriever.invoke(query)

print(f"Query: {query}")
print(f"\nTop {len(results)} retrieved chunks:")
for i, doc in enumerate(results, 1):
    print(f"\n[{i}] Source: {doc.metadata.get('source', 'unknown')}")
    print(f"    Content: {doc.page_content[:200]}...")

In [ ]:
# Similarity search with scores (lower distance = more similar)
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print("Results with similarity scores:")
for doc, score in results_with_scores:
    print(f"  Score: {score:.4f} | {doc.page_content[:80]}...")

# 🔑 Key insight: scores below ~0.3 are typically good matches (cosine distance)

---
## 5. Generation: Full RAG Chain

Combine retrieval + LLM into an end-to-end chain. The prompt explicitly tells the model to use only the provided context.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain the answer, say "I don't have information about that in my knowledge base."
Always cite which source(s) you used at the end.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    sections = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        sections.append(f"[Source: {source}]\n{doc.page_content}")
    return "\n\n---\n\n".join(sections)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built")

In [ ]:
# Test the full chain
test_questions = [
    "What is RAG and who invented it?",
    "What chunk size should I use?",
    "What is HyDE?",
    "What is the capital of France?",  # Out-of-scope — observe the fallback
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"A: {rag_chain.invoke(question)}")

---
## 6. Evaluating RAG Quality

Three key metrics for RAG evaluation:
- **Context relevance**: Did we retrieve the right chunks?
- **Faithfulness**: Is the answer grounded in the retrieved context (no hallucination)?
- **Answer relevance**: Does the answer actually address the question?

In [ ]:
from openai import OpenAI
import json

openai_client = OpenAI()

def evaluate_rag_response(question: str, context: str, answer: str) -> dict:
    """LLM-as-judge: evaluate RAG output on faithfulness and relevance."""
    eval_prompt = f"""You are a strict RAG evaluation judge. Score the following on 1-5.

Question: {question}

Retrieved Context:
{context[:800]}

Generated Answer:
{answer}

Return JSON only:
{{
  "faithfulness": <1-5>,       // Is the answer supported by the context?
  "answer_relevance": <1-5>,   // Does the answer address the question?
  "reasoning": "<1 sentence>"
}}"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": eval_prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

# Run evaluation on one example
question = "What is RAG and who invented it?"
context_docs = retriever.invoke(question)
context_text = format_docs(context_docs)
answer = rag_chain.invoke(question)

scores = evaluate_rag_response(question, context_text, answer)
print(f"Question: {question}")
print(f"Answer: {answer[:200]}...")
print(f"\nEvaluation scores:")
print(json.dumps(scores, indent=2))

---
## 7. Common Failure Modes & Fixes

| Problem | Symptom | Fix |
|---------|---------|-----|
| Wrong chunks retrieved | Irrelevant context in answer | Smaller chunks, better embeddings |
| Missing information | "I don't have info about X" (but you do) | Increase `k`, add query expansion |
| Hallucination | Claims facts not in context | Stricter prompt, lower temperature |
| Slow retrieval | High latency | Use HNSW index, cache frequent queries |
| Stale data | Outdated answers | Incremental indexing pipeline |

---

## 🧪 Exercises

1. **Add your own documents** — load a text file and index it. Ask 5 questions about it.
2. **Compare k values** — try `k=1`, `k=3`, `k=10`. How does answer quality change?
3. **Test the failure case** — ask a question that partially overlaps with your data. Does the model hallucinate or admit it doesn't know?
4. **Try HyDE** — generate a hypothetical answer to the question, then use *that* as the search query instead of the original question.

---
**Next:** [Module 03 — Fine-Tuning](../03-fine-tuning/README.md)

---
## 🧪 Exercises

1. **Chunk Size**: Index the same document at 128, 256, 512 tokens. Which gives best retrieval?
2. **HyDE vs Standard**: Compare standard retrieval vs HyDE on 10 queries.
3. **Re-ranking**: Add cross-encoder re-ranker. Measure precision@3 before/after.